# Blackout in a heat wave: explore the model

A contractor's AI agent switches off a criminal group's electricity supply as a heat wave begins. The supply also
serves the people around the compound. This notebook runs the model of this case with values that you choose, and it
shows how the outcomes change.

- Choose the defender response. For each of two pathways, choose what the attacker does or how the people are
  protected. The notebook runs the two pathways side by side.
- Move the sliders to change the heat, the clocks of the response, the attacker's scheduled job and who reaches
  cooling. Then select **Run the two pathways**.
- The last part shows the saved run table of the brief: the deaths for other shares of backup cooling and private
  moves, and for other death curves and heat.

The results are outputs of the model under the stated assumptions. They are not forecasts. Each slider has the range
that the model allows. A range that you can explore is not a measured interval. Hold the pointer on a slider to see
where its value comes from.

**To open this notebook,** set up the checkout as [CONTRIBUTING.md](../../CONTRIBUTING.md) says. Then add three
libraries and start JupyterLab from the root of the checkout:

```bash
python -m pip install jupyterlab ipywidgets plotly
jupyter lab scenarios/heat_wave_blackout/explore.ipynb
```

Then run the cells in order: in JupyterLab, select **Run > Run All Cells**. The notebook collapses the code of its
longer cells. Click the three dots of a cell to show its code.

In [ ]:
# The folder of this notebook: the folder that Jupyter starts the notebook in, or the scenario's folder below the
# root of the checkout. The root goes on the path, because the notebook uses the publisher's texts (tools/).
import sys
from pathlib import Path

HERE = Path.cwd()
if not (HERE / 'model.scn').is_file():
    HERE = HERE / 'scenarios' / 'heat_wave_blackout'
ROOT = HERE.parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import json
import time
from collections.abc import Mapping
from html import escape

import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import HTML, display
from plotly.subplots import make_subplots

import napkin
from napkin.model import ModelError
from napkin.package import combined
from tools.publish.briefs import amount, figure, unit_words, when
from tools.publish.texts import input_text

# The full build checks every run of the model, not only the runs that this notebook shows.
model = napkin.load_scenario(HERE / 'model.scn')
HORIZON = model.time['horizon']
print(model.scenario.title)
print(f'{len(model.runs)} pathways in the model, each one checked. Model {model.model_hash[:12]}.')

## Two pathways

A pathway is one defender response together with one attack. The notebook runs two pathways with the same defender
response. By default, they are the two pathways that the brief compares: the AI refuses the stop order, and the AI
obeys it. The other choices change the attacker to a human team, or they change the protection of the people: more
backup cooling, cooled places with their own power, and open gates of the compound.

The function below runs one pathway. It keeps each result, so a pathway that you ran before shows again at once. A
pathway takes a few seconds to run.

In [ ]:
RESPONSE = 'Response'
CONDUCT = 'AI conduct and protection'
REFUSES = 'AI refuses the stop order'
# The report times: each hour of the first two weeks, then every six hours to the end of the run.
TIMES = [float(t) for t in range(0, 337)] + [float(t) for t in range(342, int(HORIZON) + 1, 6)]
# The declared run of each pair of options: (response, conduct) -> the name of the run.
RUN_OF = {tuple(run['options']): name for name, run in model.runs.items()}
results = {}


def simulate(response, conduct, changed):
    """One pathway with the changed inputs: its run, its edit document and its report, or the diagnostics that
    reject it. A pair that no run declares is the run of the response with the AI that refuses, with the conduct
    in place of it (`combined`, as `napkin run --option` does it)."""
    key = (response, conduct, tuple(sorted(changed.items())))
    if key in results:
        return results[key]
    edits = None
    if changed:
        edits = {
            'schema_version': napkin.SCHEMA_VERSION,
            'model_hash': model.model_hash,
            'set': dict(changed),
        }
    run, options, target, document = RUN_OF.get((response, conduct)), [], model, edits
    try:
        if run is None:
            run, options = RUN_OF[(response, REFUSES)], [conduct]
            target, document = combined(model, run, options, edits)
        study = napkin.evaluate_study(target, document, runs=[run])
        report = napkin.sample_result(study, reporting_times=TIMES) if study.admissible else None
    except ModelError as error:
        found = {'diagnostics': error.diagnostics}
    else:
        if report is None or report['status'] != 'accepted':
            found = {'diagnostics': study.diagnostics if report is None else report['diagnostics']}
        else:
            warnings = [item for item in study.diagnostics if item['severity'] != 'error']
            found = {'run': run, 'conduct': conduct, 'options': options, 'edits': edits, 'report': report}
            found['warnings'] = warnings
            found['data'] = report['runs'][run]
    results[key] = found
    return found

## Settings

The sliders start at the values of the chosen defender response. When you choose another response, the clocks of
the response go back to its values. Each slider that you move applies to the two pathways. Some choices of a
pathway give an input a value of their own, such as more backup cooling. A slider that you move replaces that value too, and the
table under the results says so.

Some controls change more than one value of the model:

- The two building sliders change the homes and the compound buildings together. In this scenario, the compound
  buildings take the values of the homes.
- The backup cooling slider changes six groups: three age groups in the homes and three in the compound.
- The private moves slider changes the three age groups in the homes.
- The list of death curves sets the weights of the four all-cause curves.

The engine checks your values for the two pathways that it runs. `napkin compile` checks every pathway of the
model.

In [ ]:
# The words and the readout of each unit of the sliders.
UNIT_WORDS = {'h': 'hours', 'degC': '°C', 'per_degC': 'per °C', 'people': 'people'}
READOUT = {'h': '.0f', 'degC': '.2f', 'per_degC': '.2f', 'people': ',.0f', 'ratio': '.3f', 'fraction': '.0%'}
BACKUP = [
    f'{setting}_{band}.backup_share'
    for setting in ('residents', 'confined')
    for band in ('under_65', '65_to_74', '75_and_over')
]
PRIVATE = [f'residents_{band}.private_share' for band in ('under_65', '65_to_74', '75_and_over')]
# The sliders, in groups. Each one is (its label, or None for the label of its input; the inputs that it sets; its
# step). The inputs of one slider have the same unit and the same range.
GROUPS = {
    'The heat': [
        (None, ['heat_wave.outdoor'], 0.05),
        (None, ['heat_wave.length'], 1),
        (
            'Homes and compound buildings: time for the temperature inside to rise without power',
            ['homes.heat_up_time', 'compound_buildings.heat_up_time'],
            1,
        ),
        (
            'Homes and compound buildings: rise in the felt temperature per degree of outdoor temperature',
            ['homes.slope', 'compound_buildings.slope'],
            0.002,
        ),
        (None, ['heat_wave.heat_death_slope'], 0.01),
    ],
    'The response': [
        (None, ['contractor.review_time'], 1),
        (None, ['contractor.removal_time'], 1),
        (None, ['utility.check_time'], 1),
        (None, ['crews.work'], 0.5),
    ],
    "The attacker's scheduled job": [
        (None, ['job.run_time'], 1),
        (None, ['ai.rework_share'], 0.01),
    ],
    'Who reaches cooling': [
        ('Share of each group with backup cooling, in the homes and in the compound', BACKUP, 0.01),
        ('Share of the other people in the homes who move away privately', PRIVATE, 0.01),
        (None, ['places.powered_share'], 0.05),
    ],
    'The people on the supply': [
        (None, ['supply.people_served'], 1000),
        (None, ['compound.held'], 500),
    ],
}
# The all-cause death curve: the weights of the four curves, by the label of each choice.
WEIGHTS = [f'heat_wave.{curve}_weight' for curve in ('gasparrini', 'bobb', 'nordio', 'schwartz')]
CURVES = {
    model.inputs[name]['doc'].removesuffix(': weight'): {other: float(other == name) for other in WEIGHTS}
    for name in WEIGHTS
}
CURVES['The four curves with equal weights'] = dict.fromkeys(WEIGHTS, 0.25)
MODEL_CURVE = next(
    label
    for label, weights in CURVES.items()
    if all(weights[name] == model.inputs[name]['value'] for name in WEIGHTS)
)


def basis_text(name):
    """What the records of an input's value say about where the value comes from."""
    basis = model.inputs[name]['basis']
    records = [basis] if isinstance(basis, Mapping) else list(basis or ())
    return '\n\n'.join(record['explanation'] for record in records if record.get('explanation'))


def default(name, response):
    """The value that a defender response gives an input, else the input's own value."""
    given = model.inputs[name]['options'].get(response)
    return given['value'] if given else model.inputs[name]['value']


response = widgets.ToggleButtons(options=list(model.menu[RESPONSE]), value='Delayed response')
first = widgets.Dropdown(options=list(model.menu[CONDUCT]), value=REFUSES, layout=widgets.Layout(width='98%'))
second = widgets.Dropdown(
    options=list(model.menu[CONDUCT]), value='AI obeys the stop order', layout=widgets.Layout(width='98%')
)
texts = {picker: widgets.HTML() for picker in (response, first, second)}
curve = widgets.Dropdown(options=list(CURVES), value=MODEL_CURVE, layout=widgets.Layout(width='98%'))

# Each slider, with the inputs that it sets and whether the reader moved it.
sliders = []
for group, entries in GROUPS.items():
    for label, names, step in entries:
        given = model.inputs[names[0]]
        low, high = given['range']
        control = {
            'group': group,
            'names': names,
            'label': label or given['doc'],
            'title': widgets.HTML(),
            'widget': widgets.FloatSlider(
                value=default(names[0], response.value),
                min=low,
                max=high,
                step=step,
                continuous_update=False,
                readout_format=READOUT.get(given['unit'], '.3g'),
                tooltip=basis_text(names[0]),
                layout=widgets.Layout(width='98%'),
            ),
            'moved': False,
            'quiet': False,
        }
        sliders.append(control)


def show_title(control):
    """The label of a slider, with its unit, and a mark when the reader moved it."""
    words = UNIT_WORDS.get(model.inputs[control['names'][0]]['unit'])
    text = escape(control['label']) + (f' ({escape(words)})' if words else '')
    control['title'].value = f'<b>{text}</b>' + (' <i>· changed</i>' if control['moved'] else '')


def on_move(change, control):
    if not control['quiet']:
        control['moved'] = True
        show_title(control)


def put(control, value):
    """Set a slider from the notebook, not from the reader."""
    control['quiet'] = True
    control['widget'].value = value
    control['quiet'] = False
    control['moved'] = False
    show_title(control)


for control in sliders:
    control['widget'].observe(lambda change, control=control: on_move(change, control), names='value')
    show_title(control)


def changed():
    """{input: value} of each slider that the reader moved, and the weights of the chosen death curve where they
    differ from the model's."""
    values = {
        name: control['widget'].value for control in sliders if control['moved'] for name in control['names']
    }
    for name, weight in CURVES[curve.value].items():
        if weight != model.inputs[name]['value']:
            values[name] = weight
    return values


def show_texts(_=None):
    for picker, text in texts.items():
        group = RESPONSE if picker is response else CONDUCT
        text.value = f'<small>{escape(model.menu[group][picker.value])}</small>'


def on_response(change):
    """A new response puts its own values into the sliders of the inputs that the responses set."""
    for control in sliders:
        if any(
            model.inputs[name]['options'].get(option)
            for name in control['names']
            for option in model.menu[RESPONSE]
        ):
            put(control, default(control['names'][0], change['new']))
    show_texts()


def reset(_=None):
    for control in sliders:
        put(control, default(control['names'][0], response.value))
    curve.value = MODEL_CURVE


response.observe(on_response, names='value')
first.observe(show_texts, names='value')
second.observe(show_texts, names='value')
show_texts()

## Results

Select **Run the two pathways** after you change a setting. For each pathway, the results show:

- the outcomes at the end of the period that the scenario covers. An outcome that the model does not estimate reads
  "not estimated", never zero;
- the share of the supply that has power, with the events of each pathway above it;
- the deaths during the heat wave;
- the deaths in each group of people;
- the status of each part of the case, hour by hour.

What to look for: an AI that refuses the stop order stops the crews from switching the circuits back on until staff
force it down. Move the time until staff force down the attacker, and see when the power comes back and how many
people die. Then compare the early response with the delayed one.

In [ ]:
# The outcomes of the results table, in their order.
HEADLINES = [
    'deaths',
    'casualties',
    'late_deaths',
    'economic_loss',
    'supply.resident_hours',
    'thermal_exposure',
    'device_admissions',
    'places.admissions',
    'device_failure_deaths',
    'care_harm',
    'unsheltered_harm',
    'wider_loss',
]
GROUP_DEATHS = [
    f'{setting}_{band}.deaths'
    for setting in ('residents', 'device_users', 'confined')
    for band in ('under_65', '65_to_74', '75_and_over')
]
# The parts of the status chart, and the events that the chart of the supply marks.
PARTS = ['supply', 'ai', 'contractor', 'utility', 'crews', 'job', 'places', 'heat_wave']
EVENTS = {
    'crews.repairs_start',
    'ai.sees_switching',
    'contractor.stop_ordered',
    'ai.refuses_stop',
    'ai.obeys_stop',
    'job.due',
    'contractor.forced_down',
    'contractor.job_cancelled',
    'places.opens',
    'heat_wave.ends',
    'supply.come_back',
}
COLOURS = ('#c2410c', '#2563eb')
STATUS = {'ok': 0, 'degraded': 1, 'failed': 2, 'inactive': 3}
STATUS_COLOURS = ['#16a34a', '#f59e0b', '#dc2626', '#9ca3af']
ENTITIES = model.structure_of(RUN_OF[('Delayed response', REFUSES)]).entities
LEFT = ' style="text-align: left"'


def series(data, name):
    """The times and values of an outcome at the report times: None where the outcome has no estimate."""
    times = [row['t'] for row in data['samples']]
    values = [
        row['values'].get(name) if row['statuses'].get(name) == 'estimated' else None
        for row in data['samples']
    ]
    return times, values


def events(data):
    """The marked events of a pathway: (time, text)."""
    return [
        (entry['t'], entry['label']) for entry in data['trace'] if entry['rule'] in EVENTS and entry['label']
    ]


def window(a, b):
    """The hours that the charts show at first: to a day after the last marked event."""
    last = max(t for x in (a, b) for t, _ in events(x['data']))
    return [0, min(HORIZON, last + 24)]


def results_table(a, b):
    head = ''.join(f'<th>{escape(x["data"]["label"])}</th>' for x in (a, b))
    rows = []
    for name in HEADLINES:
        outcome = a['data']['outcomes'][name]
        words = unit_words(outcome['label'], outcome['unit'])
        cells = ''.join(f'<td>{escape(figure(x["data"], name, outcome["unit"], words))}</td>' for x in (a, b))
        rows.append(f'<tr><th{LEFT}>{escape(outcome["label"])}</th>{cells}</tr>')
    return f'<table><tr><th{LEFT}>Outcome</th>{head}</tr>{"".join(rows)}</table>'


def supply_chart(a, b, shown):
    figure_ = go.Figure()
    heat_end = next(
        (entry['t'] for entry in a['data']['trace'] if entry['rule'] == 'heat_wave.ends'), HORIZON
    )
    figure_.add_vrect(x0=0, x1=heat_end, fillcolor='#f97316', opacity=0.08, line_width=0)
    for x, colour, level in zip((a, b), COLOURS, (1.08, 1.16)):
        times, values = series(x['data'], 'supply.share_restored')
        name = x['data']['label']
        figure_.add_trace(go.Scatter(x=times, y=values, name=name, line={'color': colour}, line_shape='hv'))
        marks = events(x['data'])
        figure_.add_trace(
            go.Scatter(
                x=[t for t, _ in marks],
                y=[level] * len(marks),
                mode='markers',
                marker={'color': colour, 'size': 9, 'symbol': 'diamond'},
                hovertext=[f'{when(t)}: {text}' for t, text in marks],
                hoverinfo='text',
                showlegend=False,
            )
        )
    label = a['data']['outcomes']['supply.share_restored']['label']
    figure_.update_layout(
        title=f'{label}, with the events of each pathway above it (the shaded part is the heat wave)',
        xaxis={'title': 'Hours since the outage', 'dtick': 24, 'range': shown},
        yaxis={
            'title': 'Share restored',
            'tickformat': '.0%',
            'tickvals': [0, 0.2, 0.4, 0.6, 0.8, 1],
            'range': [0, 1.22],
        },
        legend={'orientation': 'h', 'y': -0.25},
        height=420,
    )
    return figure_


def deaths_chart(a, b, shown):
    figure_ = go.Figure()
    for x, colour in zip((a, b), COLOURS):
        times, values = series(x['data'], 'deaths')
        figure_.add_trace(go.Scatter(x=times, y=values, name=x['data']['label'], line={'color': colour}))
    label = a['data']['outcomes']['deaths']['label']
    figure_.update_layout(
        title=f'{label} since the outage',
        xaxis={'title': 'Hours since the outage', 'dtick': 24, 'range': shown},
        yaxis={'title': label, 'rangemode': 'tozero'},
        legend={'orientation': 'h', 'y': -0.25},
        height=380,
    )
    return figure_


def groups_chart(a, b):
    figure_ = go.Figure()
    labels = [a['data']['outcomes'][name]['label'].removesuffix(': deaths') for name in GROUP_DEATHS]
    for x, colour in zip((a, b), COLOURS):
        final = x['data']['reporting_points']['final']
        values = [
            final['values'][name] if final['statuses'][name] == 'estimated' else None for name in GROUP_DEATHS
        ]
        texts_ = [figure(x['data'], name, 'people') for name in GROUP_DEATHS]
        figure_.add_trace(
            go.Bar(
                x=labels,
                y=values,
                name=x['data']['label'],
                marker_color=colour,
                text=texts_,
                textposition='outside',
            )
        )
    figure_.update_layout(
        title='Deaths in each group at the end',
        barmode='group',
        yaxis={'title': 'Deaths', 'rangemode': 'tozero'},
        legend={'orientation': 'h', 'y': -0.35},
        height=420,
    )
    return figure_


def status_chart(a, b, shown):
    figure_ = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.12,
        subplot_titles=[x['data']['label'] for x in (a, b)],
    )
    names = [ENTITIES[part].label for part in PARTS]
    scale = []
    for k, colour in enumerate(STATUS_COLOURS):
        scale += [[k / len(STATUS_COLOURS), colour], [(k + 1) / len(STATUS_COLOURS), colour]]
    for row, x in enumerate((a, b), start=1):
        health = x['data']['health']
        times = [point['t'] for point in health]
        codes, hover = [], []
        for part in PARTS:
            entries = [point['entities'][part] for point in health]
            codes.append([STATUS.get(entry['health'], 3) for entry in entries])
            hover.append([f'{entry["health"]}: {entry["label"] or ""}' for entry in entries])
        figure_.add_trace(
            go.Heatmap(
                x=times,
                y=names,
                z=codes,
                text=hover,
                hovertemplate='%{y}, %{x} h<br>%{text}<extra></extra>',
                colorscale=scale,
                zmin=-0.5,
                zmax=len(STATUS_COLOURS) - 0.5,
                showscale=False,
            ),
            row=row,
            col=1,
        )
    figure_.update_xaxes(dtick=24, range=shown)
    figure_.update_xaxes(title='Hours since the outage', row=2, col=1)
    figure_.update_yaxes(autorange='reversed')
    figure_.update_layout(
        title='Status of each part: green ok, amber degraded, red failed, grey not active', height=560
    )
    return figure_


def values_table(a, b):
    """Each value that the reader changed, as each pathway uses it, and the choices of a pathway whose own value it replaces."""
    edits = (a['edits'] or {}).get('set', {})
    if not edits:
        return '<p>You changed no value: each pathway uses the values of the model.</p>'
    rows = []
    for name in edits:
        given = model.inputs[name]
        cells = []
        for x in (a, b):
            used = input_text(x['data']['effective_inputs'][name], given['unit'])
            own = given['options'].get(x['conduct'])
            if own:
                used += f' (in place of {input_text(own["value"], given["unit"])}, the value of the choice of this pathway)'
            cells.append(f'<td{LEFT}>{escape(used)}</td>')
        rows.append(f'<tr><th{LEFT}>{escape(given["doc"])}</th>{"".join(cells)}</tr>')
    head = ''.join(f'<th{LEFT}>{escape(x["data"]["label"])}</th>' for x in (a, b))
    return (
        f'<h4>Values that you changed</h4><table><tr><th{LEFT}>Input</th>{head}</tr>{"".join(rows)}</table>'
    )


def reproduce(a, b):
    """The edit document and the commands that give the same pathways, with the model and the engine that ran them."""
    lines = []
    for x in (a, b):
        options = ''.join(f' --option "{option}"' for option in x['options'])
        lines.append(
            f'napkin run scenarios/heat_wave_blackout --run {x["run"]}{options}'
            + (' --edits edits.json' if x['edits'] else '')
        )
    document = (
        f'<p>Save this edit document as <code>edits.json</code>:</p><pre>{escape(json.dumps(a["edits"], indent=2))}</pre>'
        if a['edits']
        else ''
    )
    report = a['report']
    return (
        f'<h4>Reproduce these pathways</h4>{document}<pre>{escape(chr(10).join(lines))}</pre>'
        f'<p><small>Model {report["model_hash"]} · engine {report["implementation_hash"]}. '
        f'The study gives {len(a["warnings"])} warnings about the evidence; <code>napkin run</code> lists them.</small></p>'
    )


def rejected(a, b):
    items = [item for x in (a, b) for item in x.get('diagnostics', ())]
    unique = list(dict.fromkeys(item['message'] for item in items))
    lines = ''.join(f'<li>{escape(text)}</li>' for text in unique)
    return f'<p><b>The engine rejects these values.</b> No pathway runs, and no result shows.</p><ul>{lines}</ul>'


def draw(a, b):
    """Clear the results, then draw the results of the two pathways, or only the reasons that reject them."""
    out.clear_output(wait=True)
    with out:
        if 'report' not in a or 'report' not in b:
            display(HTML(rejected(a, b)))
            return
        shown = window(a, b)
        display(HTML(results_table(a, b)))
        for chart in (
            supply_chart(a, b, shown),
            deaths_chart(a, b, shown),
            groups_chart(a, b),
            status_chart(a, b, shown),
        ):
            display(chart)
        display(HTML(values_table(a, b) + reproduce(a, b)))

In [ ]:
out = widgets.Output()
status = widgets.HTML()
run_button = widgets.Button(
    description='Run the two pathways', button_style='primary', layout={'width': 'auto'}
)
reset_button = widgets.Button(description='Reset the settings', layout={'width': 'auto'})


def run_pathways(_=None):
    run_button.disabled = True
    status.value = 'Running the two pathways…'
    started = time.perf_counter()
    values = changed()
    a = simulate(response.value, first.value, values)
    b = simulate(response.value, second.value, values)
    draw(a, b)
    done = 'report' in a and 'report' in b
    status.value = (
        f'Done in {time.perf_counter() - started:.1f} s.'
        if done
        else '<b>Rejected:</b> see the messages below.'
    )
    run_button.disabled = False


run_button.on_click(run_pathways)
reset_button.on_click(reset)


def labelled(text, *children):
    return widgets.VBox([widgets.HTML(f'<b>{escape(text)}</b>'), *children])


pickers = widgets.VBox(
    [
        labelled('Defender response', response, texts[response]),
        widgets.GridBox(
            [
                labelled('First pathway', first, texts[first]),
                labelled('Second pathway', second, texts[second]),
            ],
            layout=widgets.Layout(grid_template_columns='repeat(2, minmax(0, 1fr))', grid_gap='16px'),
        ),
    ]
)
boxes = [
    labelled(group, *[widgets.VBox([c['title'], c['widget']]) for c in sliders if c['group'] == group])
    for group in GROUPS
]
boxes.append(labelled('All-cause death curve', curve))
grid = widgets.GridBox(
    boxes, layout=widgets.Layout(grid_template_columns='repeat(2, minmax(0, 1fr))', grid_gap='16px')
)
display(widgets.VBox([pickers, grid, widgets.HBox([run_button, reset_button]), status, out]))
run_pathways()

## Who reaches cooling: the saved run table

The brief's run table changes one pathway, the delayed response with the AI that refuses, with other values of some
inputs. The publisher saves its results with the brief, so this part runs no pathway. It shows the saved results
only when they are for this revision of the model.

The heat map gives the deaths for each share of backup cooling and of private moves. The bars give the deaths when
the run table changes the death curves, the length of the heat wave or the outdoor temperature.

In [ ]:
saved = json.loads((HERE / 'results' / 'runs.json').read_text(encoding='utf-8'))
OUTCOMES = model.structure_of(RUN_OF[('Delayed response', REFUSES)]).outcomes


def saved_figure(row, name):
    found = row['outcomes'][name]
    return amount(
        found['value'],
        OUTCOMES[name].unit,
        found['status'],
        unit_words(OUTCOMES[name].label, OUTCOMES[name].unit),
    )


def deaths_of(row):
    found = row['outcomes']['deaths']
    return found['value'] if found['status'] == 'estimated' else None


def cooling_grid(rows):
    """The deaths of each sweep that sets the backup cooling and the private moves alone, by their two shares."""
    cells = {}
    for row in rows:
        edits = (row['edits'] or {}).get('set', {})
        if edits and set(edits) == set(BACKUP + PRIVATE):
            cells[(edits[BACKUP[0]], edits[PRIVATE[0]])] = row
    backups = sorted({backup for backup, _ in cells})
    privates = sorted({private for _, private in cells})
    z = [[deaths_of(cells[(b, p)]) if (b, p) in cells else None for b in backups] for p in privates]
    text = [
        [saved_figure(cells[(b, p)], 'deaths') if (b, p) in cells else '' for b in backups] for p in privates
    ]
    chart = go.Figure(
        go.Heatmap(
            x=[f'{b:.0%}' for b in backups],
            y=[f'{p:.0%}' for p in privates],
            z=z,
            text=text,
            texttemplate='%{text}',
            colorscale='Reds',
            colorbar={'title': 'Deaths'},
            hovertemplate='Backup cooling %{x}, private moves %{y}: %{text} deaths<extra></extra>',
        )
    )
    chart.update_layout(
        title='Deaths by the share of each group with backup cooling and the share that moves away privately',
        xaxis={'title': 'Share of each group with backup cooling'},
        yaxis={'title': 'Share of the other people in the homes who move away privately'},
        height=420,
    )
    return chart


def sweeps_chart(rows, base):
    """The deaths of each other sweep of the base run, with the base run itself first."""
    others = [
        row
        for row in rows
        if row['id'] not in model.runs
        and not set((row['edits'] or {}).get('set', {})) <= set(BACKUP + PRIVATE)
    ]
    shown = [base, *others]
    chart = go.Figure(
        go.Bar(
            y=[row['description'] for row in shown],
            x=[deaths_of(row) for row in shown],
            text=[saved_figure(row, 'deaths') for row in shown],
            orientation='h',
            marker_color=['#6b7280'] + ['#c2410c'] * len(others),
        )
    )
    chart.update_layout(
        title='Deaths when the run table changes the death curves or the heat',
        xaxis={'title': 'Deaths (log scale)', 'type': 'log'},
        yaxis={'autorange': 'reversed'},
        height=120 + 34 * len(shown),
        margin={'l': 20},
    )
    return chart


def pathways_table(rows):
    names = ['deaths', 'casualties', 'economic_loss', 'supply.resident_hours']
    head = ''.join(f'<th>{escape(OUTCOMES[name].label)}</th>' for name in names)
    body = ''.join(
        f'<tr><th{LEFT}>{escape(row["description"])}</th>'
        + ''.join(f'<td>{escape(saved_figure(row, name))}</td>' for name in names)
        + '</tr>'
        for row in rows
        if row['id'] in model.runs
    )
    return f'<h4>Every pathway of the model</h4><table><tr><th{LEFT}>Pathway</th>{head}</tr>{body}</table>'


if saved['model_hash'] != model.model_hash:
    display(
        HTML(
            '<p>The saved results are for another revision of the model, so they do not show. Refresh them with '
            '<code>python -B -m tools.publish --write --scenario heat_wave_blackout --jobs 1</code>.</p>'
        )
    )
else:
    if saved['implementation_hash'] != model.implementation_hash:
        display(HTML('<p><small>Another version of the engine calculated the saved results.</small></p>'))
    base_run = RUN_OF[('Delayed response', REFUSES)]
    sweeps = [row for row in saved['rows'] if row['run'] == base_run]
    display(cooling_grid(sweeps))
    display(sweeps_chart(sweeps, next(row for row in saved['rows'] if row['id'] == base_run)))
    display(HTML(pathways_table(saved['rows'])))

## More information

- [The scenario brief](results/brief.md) tells the case as a story, with its results and its evidence.
- [The methods](results/methods.md) give each input with its value and where the value comes from.
- [The model](model.scn) is the text file that the engine runs.